# Fit and evaluate

This notebook walks the **fit then evaluate** workflow end to end. We
generate synthetic data from a homogeneous multivariate-Poisson process
with *known* rates, fit a `PoissonProcess` to it by maximum likelihood,
check that the fitted model recovers those rates, and score it on
held-out data with TorchPP's next-event metrics.

Because the true rates are known, we can tell whether the fit worked.
The example is self-contained — there is no dataset to download.

## Setup

We fix the true generating rates (events per unit time, per type), the
observation window, the train/validation/test sizes, and the number of training
steps up front, so the whole run is reproducible.

In [ ]:
import torch

from torchpp import MVEventData, PoissonProcess, evaluate_tpp, fit
from torchpp.training import default_fit_config
from torchpp.utils import format_metrics

TRUE_RATES = [2.0, 1.0, 0.5]  # events per unit time, per type
D = len(TRUE_RATES)
T, N_TRAIN, N_VAL, N_TEST, STEPS = 20.0, 256, 64, 64, 500

## Synthetic data

A **homogeneous Poisson** process has a constant rate, which makes its
inter-arrival times exponentially distributed. So we can draw one type-`d`
stream by starting at `t = 0` and adding exponential gaps (mean `1 / rate`)
until we leave the window `T`. The gap `-log(U) / rate` for a uniform `U` is
the standard inverse-CDF draw of an exponential.

Each sequence becomes an `MVEventData`: its event times, integer marks, and
an observation window `T_obs = T`. `sort=True` orders the merged per-type
events by time.

In [ ]:
def make_poisson_sequences(n, rates, T, seed):
    """Homogeneous multivariate Poisson via exponential inter-arrival times."""
    g = torch.Generator().manual_seed(seed)
    seqs = []
    for _ in range(n):
        times, types = [], []
        for d, rate in enumerate(rates):
            t = 0.0
            while True:
                t += float(-torch.log(torch.rand((), generator=g)) / rate)
                if t > T:
                    break
                times.append(t)
                types.append(d)
        seqs.append(MVEventData(torch.tensor(times), torch.tensor(types), sort=True, T_obs=T))
    return seqs

## Build the data and the model

We generate train, validation and test sets with fixed seeds. The model is a `PoissonProcess`
over `D` event types; `terminating=False` lets each type recur (a terminating process would allow
each type at most once).

In [ ]:
train = make_poisson_sequences(N_TRAIN, TRUE_RATES, T, seed=0)
val = make_poisson_sequences(N_VAL, TRUE_RATES, T, seed=1)
test = make_poisson_sequences(N_TEST, TRUE_RATES, T, seed=2)

model = PoissonProcess(D=D, terminating=False)

## Train

The objective is the **negative log-likelihood per scored event** plus the model's
regularization term. `fit` minimises it with mini-batches, a learning-rate schedule and gradient
clipping, evaluates the validation set every `eval_freq` steps, and returns the model with the best
validation weights. One step of it is the plain PyTorch loop

```python
loss = model.loss(batch).sum() / model.num_scored_events(batch).sum() + model.regularization(batch)
loss.backward(); optimizer.step()
```

In [ ]:
config = default_fit_config(num_steps=STEPS, learning_rate=0.05, eval_freq=100, batch_size=64)
result = fit(model, train, val, config)
model = result.model
print("validation log-likelihood per event, every 100 steps:", [round(v, 4) for v in result.val_ll])

## Recovered rates

A fitted Poisson process is just its constant baseline, so the fitted rates should land close to
the ones we generated from. `baseline_values(t)` returns the rates at times `t` in the data's
time units.

In [ ]:
# The fitted constant baseline should recover the generating rates.
with torch.no_grad():
    fitted_rates = model.baseline_values(torch.tensor([0.0]))[0]
print("true rates  :", TRUE_RATES)
print("fitted rates:", [round(float(r), 3) for r in fitted_rates])

## Evaluate on held-out data

`evaluate_tpp` computes the next-event metric decomposition (joint, time and
type log-likelihood, accuracy) on the test sequences the model never saw. We pass
`compute_marginals=False` to skip the slower grid-based metrics (RMSE,
marginal accuracy, `p(d)`). `format_metrics` pretty-prints the result.

`ll_per_event` is the full-sequence log-likelihood per scored event (the training objective). It also contains the probability of no further event between the last event and the end of the window, so it is a bit lower than `joint_ll`, which splits exactly into `marginal_time_ll` + `cond_type_ll`.

In [ ]:
# Grid-free metrics; compute_marginals=False skips the slower grid-based ones.
metrics = evaluate_tpp(model, test, compute_marginals=False)
print(format_metrics(metrics))